In [1]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
import csv

In [3]:
#Setting up
unknown_token = "[UNK]"
tokenizer = Tokenizer(BPE(unk_token=unknown_token))
tokenizer.pre_tokenizer = Whitespace()

In [4]:
#Decompose CVS into a list
def read_csv_file(file):
    text_list = []
    sent_list = []
    with open(file, newline='') as csvfile:
        Train = csv.DictReader(csvfile)
        for row in Train:
            text_list.append(row['text'])
            if 'label' in row:
                sent_list.append(row['label'])
    if len(sent_list) == 0:
        return text_list
    return text_list, sent_list

In [7]:
#Initial processing
train_file = '/Users/jakeburton/Desktop/NLP/NLP grad version/3rabizi-BPE/BPE/SAMAWEL JABALLI /Train (1).csv'
test_file = '/Users/jakeburton/Desktop/NLP/NLP grad version/3rabizi-BPE/BPE/SAMAWEL JABALLI /Test (1).csv'
train_text, train_sent = read_csv_file(train_file)
test_text = read_csv_file(test_file)

In [8]:
#Normalizing the text
import re
def normalize_text(text):
    for i in range(len(text)):
        #All lowercase
        text[i] = text[i].lower()
        #Reduce repeated characters
        text[i] = re.sub(r'(.)\1{3,}', r'\1\1\1', text[i])
    return text
    

In [9]:
#Normalize the text
train_text = normalize_text(train_text)
test_text = normalize_text(test_text)

In [10]:
#Find base vocab
vocab = set()
for sentence in train_text:
    for word in sentence.split():
        vocab.add(word)
print(f"Base vocabulary size: {len(vocab)}")

Base vocabulary size: 141967


In [11]:
#Train BPE tokenizer
trainer = BpeTrainer(vocab_size=30000, special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"])
tokenizer.train_from_iterator(train_text, trainer)

In [12]:
tokenizer.save("BPE_first.json")

In [13]:
BPE_tokenizer = Tokenizer.from_file("BPE_first.json")
#Test the tokenizer
test_sentence = "This is a test sentence to check the BPE tokenizer."
encoded = BPE_tokenizer.encode(train_text[0])  
print("Tokens:", encoded.tokens)

Tokens: ['3sbaaa', 'lek', 'ou', 'le', 'se', 'im', 'riahi', 'ou', '3sbaaa', 'le', 'ca']


In [14]:
#Check fertility
fertility = 0
for sentence in test_text:
    words = sentence.split()
    tokens = BPE_tokenizer.encode(sentence).tokens
    fertility += len(tokens) / len(words)
average_fertility = fertility / len(test_text)
print(f"Average fertility: {average_fertility:.2f}")

Average fertility: 1.30
